In [16]:
pip install pandas scikit-learn


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [2]:
import pandas as pd
df = pd.read_csv('data.csv', encoding='latin-1', header=None)
df = df[[0, 5]]
df.columns = ['polarity', 'text']
print(df.head())

   polarity                                               text
0         0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1         0  is upset that he can't update his Facebook by ...
2         0  @Kenichan I dived many times for the ball. Man...
3         0    my whole body feels itchy and like its on fire 
4         0  @nationwideclass no, it's not behaving at all....


In [3]:
df = df[df.polarity != 2]

df['polarity'] = df['polarity'].map({0: 0, 4: 1})

print(df['polarity'].value_counts())

polarity
0    800000
1    800000
Name: count, dtype: int64


In [4]:
def clean_text(text):
    return text.lower()

df['clean_text'] = df['text'].apply(clean_text)

print(df[['text', 'clean_text']].head())

                                                text  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1  is upset that he can't update his Facebook by ...   
2  @Kenichan I dived many times for the ball. Man...   
3    my whole body feels itchy and like its on fire    
4  @nationwideclass no, it's not behaving at all....   

                                          clean_text  
0  @switchfoot http://twitpic.com/2y1zl - awww, t...  
1  is upset that he can't update his facebook by ...  
2  @kenichan i dived many times for the ball. man...  
3    my whole body feels itchy and like its on fire   
4  @nationwideclass no, it's not behaving at all....  


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['polarity'],
    test_size=0.2,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 1280000
Test size: 320000


In [6]:
vectorizer = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF shape (train):", X_train_tfidf.shape)
print("TF-IDF shape (test):", X_test_tfidf.shape)


TF-IDF shape (train): (1280000, 100000)
TF-IDF shape (test): (320000, 100000)


In [7]:
bnb = BernoulliNB()
bnb.fit(X_train_tfidf, y_train)

bnb_pred = bnb.predict(X_test_tfidf)

print("Bernoulli Naive Bayes Accuracy:", accuracy_score(y_test, bnb_pred))
print("\nBernoulliNB Classification Report:\n", classification_report(y_test, bnb_pred))

Bernoulli Naive Bayes Accuracy: 0.791775

BernoulliNB Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           1       0.78      0.81      0.80    160506

    accuracy                           0.79    320000
   macro avg       0.79      0.79      0.79    320000
weighted avg       0.79      0.79      0.79    320000



In [8]:
svm = LinearSVC(C=0.25, max_iter=5000)
svm.fit(X_train_tfidf, y_train)

svm_pred = svm.predict(X_test_tfidf)

print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print("\nSVM Classification Report:\n", classification_report(y_test, svm_pred))


SVM Accuracy: 0.822640625

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82    159494
           1       0.82      0.83      0.83    160506

    accuracy                           0.82    320000
   macro avg       0.82      0.82      0.82    320000
weighted avg       0.82      0.82      0.82    320000



In [9]:
logreg = LogisticRegression(C=2, max_iter=1000, solver='liblinear')
logreg.fit(X_train_tfidf, y_train)

logreg_pred = logreg.predict(X_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, logreg_pred))
print("\nLogistic Regression Classification Report:\n", classification_report(y_test, logreg_pred))


Logistic Regression Accuracy: 0.823865625

Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82    159494
           1       0.82      0.83      0.83    160506

    accuracy                           0.82    320000
   macro avg       0.82      0.82      0.82    320000
weighted avg       0.82      0.82      0.82    320000



In [10]:
print("Bernoulli Naive Bayes:",
      accuracy_score(y_test, bnb_pred))

print("SVM:",
      accuracy_score(y_test, svm_pred))

print("Logistic Regression:",
      accuracy_score(y_test, logreg_pred))

Bernoulli Naive Bayes: 0.791775
SVM: 0.822640625
Logistic Regression: 0.823865625


In [11]:
sample_tweets = ["I love this!", "I hate that!", "It was okay, not great."]
sample_vec = vectorizer.transform(sample_tweets)

print("\nSample Predictions:")
print("BernoulliNB:", bnb.predict(sample_vec))
print("SVM:", svm.predict(sample_vec))
print("Logistic Regression:", logreg.predict(sample_vec))


Sample Predictions:
BernoulliNB: [1 0 1]
SVM: [1 0 0]
Logistic Regression: [1 0 0]


In [15]:
user_sentence = input("Enter a sentence to analyze sentiment: ")
user_vec = vectorizer.transform([user_sentence])
sentiment_map = {1: "Positive", 0: "Negative"}

print("\n--- Predictions ---")

bnb_pred = bnb.predict(user_vec)[0]
print(f"BernoulliNB: {sentiment_map.get(bnb_pred, 'Unknown')}")

svm_pred = svm.predict(user_vec)[0]
print(f"SVM: {sentiment_map.get(svm_pred, 'Unknown')}")

logreg_pred = logreg.predict(user_vec)[0]
print(f"Logistic Regression: {sentiment_map.get(logreg_pred, 'Unknown')}")



--- Predictions ---
BernoulliNB: Positive
SVM: Positive
Logistic Regression: Positive
